# 🧩 Step 9 – Model Deployment with ONNX (LightGBM Models)

## 📘 1. Introduction to ONNX

### What is ONNX?
**ONNX (Open Neural Network Exchange)** is an open, cross-platform format designed to represent trained machine learning models.  
It allows models trained in one framework (e.g., LightGBM, XGBoost, PyTorch, Scikit-learn) to be **exported** and **run efficiently** on many platforms and languages using lightweight runtimes such as **ONNX Runtime**.

ONNX separates the *training* environment from the *serving* environment — you train once, then deploy anywhere.

---

## 🔍 Why Use ONNX?

| Situation | Benefit of ONNX |
|------------|----------------|
| **High-volume or real-time predictions** | ONNX Runtime uses graph optimizations and efficient kernels, often reducing latency by 30–70%. |
| **Cross-platform deployment** | Serve models in C++, C#, Java, Go, Rust, or mobile apps — without retraining. |
| **Lightweight inference environments** | No need to install heavy Python packages like LightGBM or scikit-learn in production containers. |
| **Edge devices / low resources** | Models can be quantized for smaller size and faster inference. |
| **Model consistency** | ONNX ensures identical predictions across environments using the same model graph. |

---

## 🧠 Why ONNX for this project?

In this project, we trained multiple models and then the best model for each day in a **multi-horizon temperature forecasting** (1-day → 5-day ahead) for Hanoi.
 
By exporting to ONNX, we can:
- Deploy all 5 forecasting models in **a single portable format**,  
- Achieve faster inference (especially for CPU-based servers or IoT devices),  
- Use ONNX Runtime to serve forecasts without needing the training dependencies.

## ⚙️ 2. Environment Setup

In [4]:
# Install required libraries (run once)
# !pip install onnxmltools 
# !pip installonnx onnxruntime lightgbm (preferably install in shell)
# !pip install skl2onnx

In [5]:
import os
import joblib
import numpy as np
import pandas as pd
import onnx
import onnxruntime as ort
import onnxmltools
from onnxmltools.convert.common.data_types import FloatTensorType
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType as SklFloatTensorType
from warnings import filterwarnings
filterwarnings("ignore")

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


## 🧱 3. Convert Models (Day 1 → Day 5) to ONNX

In [8]:
### Paths and Model Loading ###
###############################
base_dir = "../models/hourly_trained/"
onnx_dir = os.path.join(base_dir, "onnx_exports")
os.makedirs(onnx_dir, exist_ok=True)

# Load supporting files
feature_columns = joblib.load(os.path.join(base_dir, "feature_columns_5day.joblib"))

# Load models
model_files = {
    '1': "best_model_day_1_xgboost.joblib",
    '2': "best_model_day_2_lightgbm.joblib",
    '3': "best_model_day_3_xgboost.joblib",
    '4': "best_model_day_4_xgboost.joblib",
    '5': "best_model_day_5_xgboost.joblib",
}

models = {day: joblib.load(os.path.join(base_dir, fname)) for day, fname in model_files.items()}
print("✅ All models loaded successfully")


✅ All models loaded successfully


In [9]:
X_test = pd.read_csv("../data/processed/hourly_X_test.csv")
X_test = X_test[feature_columns].astype(np.float32)

# Take a small sample for ONNX shape inference
X_sample = X_test[:10].astype(np.float32)
initial_type = [('float_input', FloatTensorType([None, X_sample.shape[1]]))]
initial_type_skl = [('float_input', SklFloatTensorType([None, X_sample.shape[1]]))]

print("✅ X_test shape:", X_test.shape)

✅ X_test shape: (693, 91)


In [10]:
### Convert each model to ONNX ###
##################################
onnx_models = {}

for day, model in models.items():
    print(f"🔄 Converting Day {day} model to ONNX...")

    if 'xgboost' in model_files[day]:
        onnx_model = onnxmltools.convert_xgboost(model.get_booster(), initial_types=initial_type)

    elif 'lightgbm' in model_files[day]:
        onnx_model = onnxmltools.convert_lightgbm(model.booster_, initial_types=initial_type)
    
    else:  # Gradient Boosting
        onnx_model = convert_sklearn(model, initial_types=initial_type_skl)

    save_path = os.path.join(onnx_dir, f"day_{day}_model.onnx")
    onnx.save_model(onnx_model, save_path)
    onnx_models[day] = save_path
    print(f"✅ Day {day} model converted and saved to {save_path}")

🔄 Converting Day 1 model to ONNX...
✅ Day 1 model converted and saved to ../models/hourly_trained/onnx_exports\day_1_model.onnx
🔄 Converting Day 2 model to ONNX...
✅ Day 2 model converted and saved to ../models/hourly_trained/onnx_exports\day_2_model.onnx
🔄 Converting Day 3 model to ONNX...
✅ Day 3 model converted and saved to ../models/hourly_trained/onnx_exports\day_3_model.onnx
🔄 Converting Day 4 model to ONNX...
✅ Day 4 model converted and saved to ../models/hourly_trained/onnx_exports\day_4_model.onnx
🔄 Converting Day 5 model to ONNX...
✅ Day 5 model converted and saved to ../models/hourly_trained/onnx_exports\day_5_model.onnx


## 🚀 4. Running Inference with ONNX Runtime
We’ll now load each exported ONNX model and compare its predictions
to the original LightGBM model to confirm consistency.

In [11]:
for day, model in models.items():
    print(f"📈 Comparing predictions for Day {day}")

    # Original model predictions
    skl_pred = model.predict(X_test)

    # ONNX predictions
    sess = ort.InferenceSession(onnx_models[day], providers=['CPUExecutionProvider'])
    input_name = sess.get_inputs()[0].name
    onnx_pred = sess.run(None, {input_name: X_test.values.astype(np.float32)})[0].ravel()

    # Compare
    diff = np.abs(skl_pred - onnx_pred).mean()
    print(f'✅ Mean absolute difference: {diff:.6f}')

📈 Comparing predictions for Day 1
✅ Mean absolute difference: 0.000006
📈 Comparing predictions for Day 2
✅ Mean absolute difference: 0.035477
📈 Comparing predictions for Day 3
✅ Mean absolute difference: 0.000005
📈 Comparing predictions for Day 4
✅ Mean absolute difference: 0.000007
📈 Comparing predictions for Day 5
✅ Mean absolute difference: 0.000011


## ✅ 5. Deployment Summary

- 5 ONNX models exported:
  - `day_1_model.onnx` (XGBoost)
  - `day_2_model.onnx` (LightGBM)
  - `day_3_model.onnx` (LightGBM)
  - `day_4_model.onnx` (GradientBoosting)
  - `day_5_model.onnx` (XGBoost)

- Saved to: `../models/hourly_trained/onnx_exports/`
- Each model verified against original Python prediction.
- Input features standardized using `scaler_5day.joblib` and `feature_columns_5day.joblib`.

We can now deploy these ONNX models using `onnxruntime` for fast, lightweight inference.
